# azure_keyvault_secrets: 5 Basic Usage Examples

**Note:** These examples run the real `SecretClient` against an in-notebook stub of the Key Vault REST API, so no Key Vault or credentials are needed.

In [ ]:
# The stub vault: SecretClient sends real HTTP requests through azure-core's RequestsTransport, and
# CannedAdapter answers them from the `vault` dict instead of the network. The first request for a
# vault carries no token; the 401 challenge tells the client which tenant and resource to ask a token
# for, and FakeCredential hands out a fixed one.
import http
import io
import json
import time
from urllib.parse import urlsplit

import requests
from azure.core.credentials import AccessToken
from azure.core.pipeline.transport import RequestsTransport
from azure.keyvault.secrets import SecretClient

VAULT_URL = "https://example-vault.vault.azure.net"
VERSION = "0123456789abcdef0123456789abcdef"
CHALLENGE = {"WWW-Authenticate": 'Bearer authorization="https://login.microsoftonline.com/'
                                 '00000000-0000-0000-0000-000000000000", resource="https://vault.azure.net"'}
vault = {"database-password": "example-value-1", "api-key": "example-value-2"}
deleted = {}


class CannedAdapter(requests.adapters.BaseAdapter):
    def __init__(self, handler):
        super().__init__()
        self.handler = handler  # request -> (status code, JSON payload, extra headers)
        self.requests = []

    def send(self, request, **kwargs):
        self.requests.append(request)
        status_code, payload, headers = self.handler(request)
        body = json.dumps(payload).encode()
        response = requests.Response()
        response.status_code = status_code
        response.reason = http.HTTPStatus(status_code).phrase
        response.headers = requests.structures.CaseInsensitiveDict(
            {"Content-Type": "application/json", "Content-Length": str(len(body)), **headers})
        response.raw = io.BytesIO(body)
        response.url = request.url
        response.request = request
        return response

    def close(self):
        pass


def secret_bundle(name, value=None, **extra):
    bundle = {"id": f"{VAULT_URL}/secrets/{name}/{VERSION}",
              "attributes": {"enabled": True, "created": 1700000000, "updated": 1700000000,
                             "recoveryLevel": "Recoverable+Purgeable"}, **extra}
    if value is not None:
        bundle["value"] = value
    return bundle


def key_vault(request):
    if "Authorization" not in request.headers:
        return 401, {}, CHALLENGE
    parts = urlsplit(request.url).path.strip("/").split("/")
    if parts == ["secrets"]:
        return 200, {"value": [secret_bundle(name) for name in sorted(vault)], "nextLink": None}, {}
    collection, name = parts[0], parts[1]
    deletion = {"recoveryId": f"{VAULT_URL}/deletedsecrets/{name}",
                "deletedDate": 1700000100, "scheduledPurgeDate": 1707776100}
    if collection == "secrets" and request.method == "PUT":
        vault[name] = json.loads(request.body)["value"]
    if collection == "secrets" and request.method == "DELETE" and name in vault:
        deleted[name] = vault.pop(name)
        return 200, secret_bundle(name, **deletion), {}
    if collection == "deletedsecrets" and name in deleted:
        return 200, secret_bundle(name, **deletion), {}
    if collection == "secrets" and name in vault:
        return 200, secret_bundle(name, vault[name]), {}
    return 404, {"error": {"code": "SecretNotFound",
                           "message": f"A secret with (name/id) {name} was not found in this key vault."}}, {}


class FakeCredential:
    def __init__(self):
        self.scopes = []

    def get_token(self, *scopes, **kwargs):
        self.scopes.append(scopes)
        return AccessToken("fake-token", int(time.time()) + 3600)

In [ ]:
adapter = CannedAdapter(key_vault)
session = requests.Session()
session.mount("https://", adapter)
credential = FakeCredential()
client = SecretClient(VAULT_URL, credential=credential, transport=RequestsTransport(session=session))
assert client.vault_url == VAULT_URL

In [ ]:
# Example 1: get a secret
secret = client.get_secret("database-password")
assert (secret.name, secret.value) == ("database-password", "example-value-1")
assert secret.properties.version == VERSION and secret.properties.enabled is True

# The first request was answered with the 401 challenge; the retry carried the bearer token
first, retry = adapter.requests[:2]
assert "Authorization" not in first.headers
assert retry.headers["Authorization"] == "Bearer fake-token"
assert credential.scopes == [("https://vault.azure.net/.default",)], credential.scopes
print(f"Retrieved secret {secret.name!r}, version {secret.properties.version}")

In [ ]:
# Example 2: set a secret
created = client.set_secret("new-password", "example-value-3", content_type="text/plain")
assert (created.name, created.value) == ("new-password", "example-value-3")
assert vault["new-password"] == "example-value-3"
assert client.get_secret("new-password").value == "example-value-3"
print("Secret 'new-password' created")

In [ ]:
# Example 3: list the secrets
names = [properties.name for properties in client.list_properties_of_secrets()]
assert names == ["api-key", "database-password", "new-password"], names
print("Secrets in vault:", names)

In [ ]:
# Example 4: a missing secret raises ResourceNotFoundError
from azure.core.exceptions import ResourceNotFoundError

try:
    client.get_secret("non-existent-secret")
except ResourceNotFoundError as err:
    assert err.status_code == 404 and err.error.code == "SecretNotFound", (err.status_code, err.error)
    print(f"Caught expected error: {err.error.code}")
else:
    raise AssertionError("get_secret('non-existent-secret') did not raise ResourceNotFoundError")

In [ ]:
# Example 5: delete a secret
deleted_secret = client.begin_delete_secret("api-key").result()
assert deleted_secret.name == "api-key"
assert deleted_secret.recovery_id == f"{VAULT_URL}/deletedsecrets/api-key"
assert deleted_secret.deleted_date is not None and deleted_secret.scheduled_purge_date is not None
assert "api-key" not in vault and deleted["api-key"] == "example-value-2"
assert [p.name for p in client.list_properties_of_secrets()] == ["database-password", "new-password"]
print(f"Deleted {deleted_secret.name!r} on {deleted_secret.deleted_date:%Y-%m-%d}")